# 18 — Readiness ML/RAG avant production

Ce notebook consolide la qualité du RAG avant la construction de l'API, de Docker et du monitoring. Il ne relance ni embeddings, ni cross-encoder, ni appels OpenAI : il lit uniquement les artefacts versionnés des expériences finales.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'notebooks' else CURRENT_DIR
REPORT_ROOT = PROJECT_ROOT / 'reports' / 'generated' / 'multidoc2dial_v1'

def load(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

optimization = load(REPORT_ROOT / 'preproduction_optimization_experiments' / 'preproduction_rag_optimization_v1_0ec70b167d' / 'preproduction_optimization_report.json')
rescue = load(REPORT_ROOT / 'preproduction_rescue_reranking_experiments' / 'preproduction_rescue_reranking_v1_bc9cfd9b21' / 'preproduction_rescue_reranking_report.json')
context = load(REPORT_ROOT / 'preproduction_context_experiments' / 'preproduction_context_validation_v1_d5565ad046' / 'preproduction_context_report.json')
action = load(REPORT_ROOT / 'answer_action_planner_experiments' / 'answer_action_planner_v1_35a93db7af' / 'answer_action_planner_report.json')
evidence = load(REPORT_ROOT / 'answer_evidence_planner_experiments' / 'answer_evidence_planner_v1_0def41dfa1' / 'answer_evidence_planner_report.json')
compatibility = load(REPORT_ROOT / 'evidence_action_compatibility_experiments' / 'evidence_action_compatibility_v1_d5a61d1988' / 'evidence_action_compatibility_report.json')
evidence_l6 = load(REPORT_ROOT / 'answer_evidence_reranking_experiments' / 'answer_evidence_reranking_v1_25bcf264db' / 'answer_evidence_reranking_report.json')
contract = load(REPORT_ROOT / 'planned_generation_contract_experiments' / 'planned_generation_contract_v1_28e48a64b7' / 'planned_generation_contract_report.json')
planned_canary = load(REPORT_ROOT / 'planned_generation_canary_experiments' / 'planned_generation_two_case_canary_v1_18945bec85' / 'planned_generation_canary_report.json')
fresh_human = load(REPORT_ROOT / 'fresh_human_canary_experiments' / 'fresh_human_generation_canary_v1_99eefa6747' / 'fresh_human_canary_evaluation_report.json')
readiness = load(REPORT_ROOT / 'preproduction_readiness' / 'preproduction_ml_readiness_report.json')

assert optimization['gates_passed']
assert rescue['gates_passed']
assert context['quality']['gates_passed']
assert action['gates_passed']
assert evidence['gates_passed']
assert compatibility['gates_passed']
assert contract['gates_passed']
assert planned_canary['gates_passed']
assert fresh_human['human_review_status'] == 'complete'
assert evidence_l6['gates_passed'] is False
assert readiness['evaluation_protocol']['test_split_loaded'] is False
print('Artefacts chargés ; split test ouvert : False ; appels OpenAI dans cette étape : 0')

Artefacts chargés ; split test ouvert : False ; appels OpenAI dans cette étape : 0


## 1. Retrieval et contexte

Le contrôleur conversationnel reconstruit une requête autonome, estime la confiance du retrieval et déclenche un secours lexical uniquement lorsque nécessaire. Le contexte compact reste limité à 5 000 tokens. Ces composants ont été calibrés sur `development`, puis mesurés séparément sur le holdout.

In [2]:
display(pd.DataFrame([
    {'variant': 'retrieval baseline', **optimization['baseline']['holdout']},
    {'variant': 'retrieval optimized', **optimization['optimized']['holdout']},
]))
display(pd.DataFrame([
    {'variant': 'context baseline', **context['metrics']['holdout']},
    {'variant': 'context optimized', **context['metrics']['holdout']},
]))

,variant,recall_at_5,recall_at_10,recall_at_20,recall_at_50,mrr_at_10
0,retrieval baseline,0.758041,0.845029,0.907164,0.951023,0.568894
1,retrieval optimized,0.758041,0.845029,0.917398,0.956140,0.568894


,variant,examples,baseline_gold_span_recall,optimized_gold_span_recall,gold_span_recall_delta,rescued_examples,lost_examples,unchanged_examples,mean_baseline_context_tokens,mean_optimized_context_tokens,max_optimized_context_tokens,mean_optimized_context_chunks,rescue_trigger_rate
0,context baseline,1368,0.819444,0.841374,0.02193,38,8,1322,4853.643275,4853.817251,4996,9.106725,0.227339
1,context optimized,1368,0.819444,0.841374,0.02193,38,8,1322,4853.643275,4853.817251,4996,9.106725,0.227339


## 2. Planificateur d'action sélectif

Le problème restant n'était plus seulement de retrouver le bon document : après une réponse comme « Yes », le système devait savoir s'il fallait répondre ou poser la prochaine question. Un classifieur local combine l'état conversationnel et la source utilisée lors de la question précédente. Il ne force pas les cas ambigus : il choisit `answer`, `ask_followup` ou `defer_for_clarification`.

Le modèle est entraîné sur le split `train`. La calibration et le holdout de validation sont séparés par `conversation_id`, ce qui empêche deux tours d'une même conversation de contaminer les deux partitions.

In [3]:
display(pd.DataFrame([action['metrics']['development_selective'], action['metrics']['holdout_selective']], index=['development','holdout']))
display(pd.DataFrame(action['hard_regression_cases'])[['example_id','predicted_action','expected_action','ask_followup_probability','correct']])

,count,accepted_count,deferred_count,coverage,accepted_accuracy,unsafe_error_count,unsafe_error_rate,answer_count,ask_followup_count,answer_probability_max,ask_followup_probability_min
development,577,438,139,0.759099,0.888128,49,0.084922,351,87,0.3,0.75
holdout,232,187,45,0.806034,0.882353,22,0.094828,153,34,0.3,0.75


,example_id,predicted_action,expected_action,ask_followup_probability,correct
0,eval_3f8341a5f07f54f3e0aa4d73,answer,answer,0.23665,True
1,eval_a730016c76c3f9699354f9ef,ask_followup,ask_followup,0.95000,True


## 3. Navigation vers les spans atomiques

Le TF‑IDF seul sur les spans a été insuffisant. La stratégie retenue suit d'abord la structure du document depuis le span cité par la dernière question : après une réponse positive, elle examine les nœuds suivants ; après une réponse négative, elle privilégie les nœuds précédents puis les alternatives proches. Le TF‑IDF ne sert qu'à compléter le classement. Cette navigation ne lit ni le gold ni la réponse de référence au runtime.

In [4]:
display(pd.DataFrame([evidence['metrics']['development'], evidence['metrics']['holdout']], index=['development','holdout']))
display(pd.DataFrame(evidence['hard_regression_cases']))

,count,gold_available_count,gold_availability_rate,conditional_mrr,conditional_recall_at_5,conditional_recall_at_8,conditional_recall_at_12,conditional_recall_at_16,conditional_recall_at_20
development,437,409,0.935927,0.646659,0.757946,0.833741,0.855746,0.877751,0.892421
holdout,186,170,0.913978,0.736672,0.829412,0.900000,0.917647,0.935294,0.947059


,example_id,planner_action,gold_available_in_context,first_gold_rank,gold_in_shortlist
0,eval_3f8341a5f07f54f3e0aa4d73,answer,True,1,True
1,eval_a730016c76c3f9699354f9ef,ask_followup,True,1,True


## 4. Expérience cross-encoder rejetée

Le L6 a été testé localement sur les 12 spans. Il améliore légèrement Recall@3 mais réduit le MRR, donc il échoue au quality gate. Le classement structuré, plus simple et moins coûteux, reste la version sélectionnée.

In [5]:
display(pd.DataFrame([
    {'variant':'structured baseline', **evidence_l6['metrics']['holdout_baseline']},
    {'variant':'L6 + RRF challenger', **evidence_l6['metrics']['holdout_selected']},
]))
display(pd.DataFrame([evidence_l6['gate_results']]).T.rename(columns={0:'passed'}))

,variant,count,gold_available_count,conditional_mrr,conditional_recall_at_1,conditional_recall_at_3,conditional_recall_at_5,conditional_recall_at_12
0,structured baseline,186,170,0.733912,0.658824,0.782353,0.829412,0.917647
1,L6 + RRF challenger,186,170,0.726984,0.647059,0.794118,0.829412,0.917647


,passed
holdout_recall_at_3_nonregression,True
holdout_mrr_nonregression,False
holdout_recall_at_12_nonregression,True
hard_regression_gold_spans_in_top_3,True
score_coverage,True
test_split_locked,True


## 5. Compatibilité entre l'action et la preuve

Un classifieur local apprend sur `train` quels spans représentent une réponse et lesquels représentent une condition. Il améliore le classement, puis une règle calibrée autorise une preuve primaire uniquement pour les réponses positives dont la compatibilité est suffisamment élevée. Sur holdout, cette preuve primaire couvre 49,46 % des cas avec 86,96 % de précision. Les réponses négatives ne reçoivent jamais automatiquement cette priorité.

In [6]:
display(pd.DataFrame([
    {'variant':'structured baseline', **compatibility['metrics']['holdout_baseline']},
    {'variant':'action-compatible', **compatibility['metrics']['holdout_selected']},
]))
display(pd.DataFrame(compatibility['hard_regression_cases']))

,variant,count,gold_available_count,recall_at_1,recall_at_3,recall_at_5,mrr
0,structured baseline,186,170,0.658824,0.782353,0.829412,0.733912
1,action-compatible,186,170,0.670588,0.800000,0.870588,0.749314


,example_id,primary_evidence_accepted,primary_evidence_correct,primary_evidence_confidence,first_span_id
0,eval_3f8341a5f07f54f3e0aa4d73,True,True,0.87994,span_7b344d8c222c5d720a180cc5
1,eval_a730016c76c3f9699354f9ef,True,True,0.95000,span_68b4eedbb1e45822c2cde5ab


## 6. Contrat de génération verrouillé

Le LLM n'a plus à décider librement du comportement. La requête lui impose `answer` ou `ask_followup` et lui fournit 12 spans atomiques identifiés `E1…E12`. La sortie suit un JSON Schema strict avec `status`, `answer` et `citations`. Chaque citation doit contenir un identifiant existant et une citation textuelle exacte. Les spans sont isolés comme données non fiables pour réduire le risque de prompt injection.

In [7]:
display(pd.DataFrame([contract['token_budget']]))
display(pd.DataFrame([contract['evidence_quality']]))
display(pd.DataFrame(contract['hard_regression_cases']))
print('Requêtes prêtes :', contract['data_protocol']['generation_requests_prepared'])
print('Déférences sans génération :', contract['data_protocol']['planner_deferrals'])
print('Appels OpenAI effectués :', contract['data_protocol']['openai_calls_made'])

,configured_maximum,mean,p95,maximum,violations
0,2200,1293.879615,1522.0,1774,0


,holdout_gold_available_count,holdout_gold_in_request_rate_when_available,evidence_depth
0,170,0.917647,12


,example_id,expected_action,actual_action,first_gold_rank,passed
0,eval_3f8341a5f07f54f3e0aa4d73,answer,answer,1,True
1,eval_a730016c76c3f9699354f9ef,ask_followup,ask_followup,1,True


Requêtes prêtes : 623
Déférences sans génération : 176
Appels OpenAI effectués : 0


## 7. Canary ciblé final

Le canary exécute les deux anciennes régressions avec le contrat verrouillé. Le validateur exige le statut correct, un JSON conforme, des citations textuelles exactes et les termes attendus. Pour FAFSA, il interdit aussi les alternatives générales afin d'exiger uniquement la question sur l'incarcération.

In [8]:
display(pd.DataFrame([planned_canary['metrics']]))
display(pd.DataFrame(planned_canary['cases'])[['example_id','contract_valid','citations_valid','status_correct','required_terms_present','targeted_regression_pass']])
assert planned_canary['metrics']['targeted_regression_pass_rate'] == 1.0

,completion_rate,contract_valid_rate,targeted_regression_pass_rate
0,1.0,1.0,1.0


,example_id,contract_valid,citations_valid,status_correct,required_terms_present,targeted_regression_pass
0,eval_3f8341a5f07f54f3e0aa4d73,True,True,True,True,True
1,eval_a730016c76c3f9699354f9ef,True,True,True,True,True


## 8. Fresh human canary

Vingt nouveaux cas du holdout, cinq par domaine, ont été évalués manuellement sans ouvrir le split test. Un cas passe de bout en bout seulement si la décision d'action, la faithfulness et les citations passent, avec une correctness humaine d'au moins 3/4. Les seuils proviennent des configurations de qualité antérieures au test.

In [9]:
display(pd.DataFrame([fresh_human['human_metrics']]))
domain_rows = []
for domain, values in fresh_human['metrics_by_domain'].items():
    domain_rows.append({
        'domain': domain,
        'examples': values['examples'],
        'overall_pass_rate': values['overall_pass_rate'],
        'action_pass_rate': values['action_decision_pass_rate'],
        'faithfulness_pass_rate': values['faithfulness_pass_rate'],
        'citation_pass_rate': values['citation_correctness_pass_rate'],
        'correctness_mean_0_to_4': values['correctness_mean_0_to_4'],
    })
display(pd.DataFrame(domain_rows))
display(pd.DataFrame([fresh_human['gate_results']]).T.rename(columns={0:'passed'}))
assert fresh_human['gates_passed'] is False

,examples,action_decision_pass_count,action_decision_pass_rate,faithfulness_pass_count,faithfulness_pass_rate,citation_correctness_pass_count,citation_correctness_pass_rate,correctness_mean_0_to_4,correctness_distribution,correctness_acceptable_count,correctness_acceptable_rate,overall_pass_count,overall_pass_rate,hallucination_count,hallucination_rate,wilson_95
0,20,16,0.8,18,0.9,18,0.9,2.8,"{'0': 1, '1': 4, '2': 4, '3': 0, '4': 11}",11,0.55,11,0.55,2,0.1,{'action_decision_pass_rate': [0.5839825677481...


,domain,examples,overall_pass_rate,action_pass_rate,faithfulness_pass_rate,citation_pass_rate,correctness_mean_0_to_4
0,dmv,5,0.6,0.8,1.0,1.0,3.2
1,ssa,5,0.6,1.0,1.0,1.0,2.8
2,studentaid,5,0.0,0.4,0.6,0.6,1.2
3,va,5,1.0,1.0,1.0,1.0,4.0


,passed
annotation_completeness,True
automatic_contract_valid_rate,False
automatic_citation_valid_rate,False
human_overall_pass_rate,False
human_faithfulness_pass_rate,False
human_citation_correctness_pass_rate,False
human_hallucination_rate,False
test_split_locked,True


## 9. Décision

Les composants hors ligne et les deux anciennes régressions passent, mais le test humain frais échoue. Le système ne doit donc pas être figé pour le test ni promu. Le split test reste verrouillé pendant une analyse ciblée des problèmes observés, principalement Student Aid et la complétude des réponses.

In [10]:
display(pd.DataFrame(readiness['components']).T)
display(Markdown(f"**Statut global : {readiness['overall_status']}**"))
display(pd.DataFrame([readiness['remaining_gate']]))
assert readiness['overall_status'] == 'fresh_human_canary_failed'
assert readiness['production_promotion_authorized'] is False

,status,report,note
conversation_controller_and_calibrated_rescue,passed,reports\generated\multidoc2dial_v1\preproducti...,None
local_cross_encoder_rescue_reranker,passed,reports\generated\multidoc2dial_v1\preproducti...,None
optimized_compact_5k_context,passed,reports\generated\multidoc2dial_v1\preproducti...,None
selective_answer_action_planner,passed,reports\generated\multidoc2dial_v1\answer_acti...,None
structured_atomic_evidence_planner,passed,reports\generated\multidoc2dial_v1\answer_evid...,None
evidence_action_compatibility_and_primary_gate,passed,reports\generated\multidoc2dial_v1\evidence_ac...,None
optional_atomic_evidence_cross_encoder,rejected_retained_structured_baseline,reports\generated\multidoc2dial_v1\answer_evid...,"Recall@3 increased slightly, but MRR regressed..."
planned_generation_contract,passed,reports\generated\multidoc2dial_v1\planned_gen...,None
planned_generation_canary,passed,reports\generated\multidoc2dial_v1\planned_gen...,Two targeted regressions passed under the stri...
fresh_human_canary,failed,reports\generated\multidoc2dial_v1\fresh_human...,Independent review of 20 newly sampled holdout...


**Statut global : fresh_human_canary_failed**

,name,reason,failed_gates,test_split_remains_locked
0,fresh_human_canary_remediation,"The fresh human canary failed, especially on S...","[automatic_contract_valid_rate, automatic_cita...",True
